# 03 Modeling & Evaluation
**AR Risk Scoring Project | Finance Analytics**

Goal: train a baseline (Logistic Regression) and a final model (Random Forest),
evaluate with ROC-AUC, explain with SHAP, and produce the risk score output.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install",
                "pandas", "numpy", "matplotlib", "seaborn",
                "scikit-learn", "shap", "joblib", "-q"])


In [ ]:
import warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, classification_report, RocCurveDisplay,
    ConfusionMatrixDisplay, precision_recall_curve, average_precision_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120

REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)

FEATURE_COLS = [
    "credit_limit","age","debt_ratio","monthly_income",
    "num_open_credit_lines","num_real_estate_loans","num_dependents",
    "weighted_late_score","utilisation_ratio","income_band","dependents_income_ratio",
    "has_any_late","log_income","age_risk_flag","credit_lines_per_loan","high_debt_and_late",
]
TARGET = "target"

df = pd.read_csv("../data/processed/features_final.csv").fillna(0)
X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Default rate — Train: {y_train.mean():.2%} | Test: {y_test.mean():.2%}")


## 1. Baseline Logistic Regression

In [ ]:
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

cv_lr = cross_val_score(lr_pipe, X_train, y_train,
                        cv=StratifiedKFold(5), scoring="roc_auc", n_jobs=-1)
print(f"Logistic Regression | CV AUC: {cv_lr.mean():.4f} ± {cv_lr.std():.4f}")

lr_pipe.fit(X_train, y_train)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]
print(f"Logistic Regression | Test AUC: {roc_auc_score(y_test, lr_proba):.4f}")
print("\n" + classification_report(y_test, (lr_proba>=0.5).astype(int),
                                    target_names=["On Time","Default"]))


## 2. Final Model Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    class_weight="balanced", random_state=42, n_jobs=-1,
)

cv_rf = cross_val_score(rf, X_train, y_train,
                        cv=StratifiedKFold(5), scoring="roc_auc", n_jobs=-1)
print(f"Random Forest | CV AUC: {cv_rf.mean():.4f} ± {cv_rf.std():.4f}")

rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
print(f"Random Forest | Test AUC: {roc_auc_score(y_test, rf_proba):.4f}")
print("\n" + classification_report(y_test, (rf_proba>=0.5).astype(int),
                                    target_names=["On Time","Default"]))


## 3. ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
RocCurveDisplay.from_predictions(y_test, lr_proba, ax=axes[0], name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, rf_proba, ax=axes[0], name="Random Forest")
axes[0].plot([0,1],[0,1],"k--", linewidth=1, label="Random baseline")
axes[0].set_title("ROC Curve", fontsize=12, fontweight="bold")
axes[0].legend()

# Precision-Recall
for proba, name, color in [(lr_proba,"Logistic Regression","#4C9BE8"),
                            (rf_proba,"Random Forest","#E8674C")]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, color=color, linewidth=2, label=f"{name} (AP={ap:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve", fontsize=12, fontweight="bold")
axes[1].legend()

plt.suptitle("Model Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS / "roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/roc_pr_curves.png")


## 4. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, model, proba, name in [
    (axes[0], lr_pipe, lr_proba, "Logistic Regression"),
    (axes[1], rf,      rf_proba, "Random Forest"),
]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, (proba >= 0.5).astype(int),
        display_labels=["On Time","Default"],
        cmap="Blues", ax=ax, colorbar=False,
    )
    ax.set_title(name, fontsize=12, fontweight="bold")

plt.suptitle("Confusion Matrix (threshold = 0.5)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. SHAP Feature Importance & Explainability

In [ ]:
explainer   = shap.TreeExplainer(rf)
sample      = X_test.sample(800, random_state=42)
shap_values = explainer.shap_values(sample)

# summary plot
fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_values[:, :, 1], sample,
                  feature_names=FEATURE_COLS, show=False)
plt.title("SHAP Feature Importance — Default Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_summary.png")


In [ ]:
# SHAP bar plot (mean |SHAP|)
shap_mean = np.abs(shap_values[:, :, 1]).mean(axis=0)
shap_df = pd.Series(shap_mean, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(shap_df.index, shap_df.values, color="#4C9BE8", edgecolor="white")
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Global Feature Importance (SHAP)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS / "shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Risk Score Output

In [ ]:
# Attach risk scores to test set and segment into tiers
result = X_test.copy()
result["risk_score"]  = rf_proba
result["actual"]      = y_test.values
result["risk_tier"]   = pd.cut(
    rf_proba,
    bins=[0, 0.20, 0.40, 0.60, 1.0],
    labels=["Low", "Medium", "High", "Critical"],
)

print("Risk tier distribution:")
tier_stats = result.groupby("risk_tier", observed=True).agg(
    customers=("risk_score","count"),
    default_rate=("actual","mean"),
    avg_score=("risk_score","mean"),
)
tier_stats["default_rate"] = (tier_stats["default_rate"]*100).round(2)
tier_stats["avg_score"]    = tier_stats["avg_score"].round(4)
print(tier_stats.to_string())

out_path = Path("../data/processed/risk_scores.csv")
result[["risk_score","risk_tier","actual"]].to_csv(out_path, index=False)
print(f"\nRisk scores saved → {out_path}")


## 7. Save Model

In [ ]:
model_path = REPORTS / "rf_model.joblib"
joblib.dump(rf, model_path)
print(f"Model saved → {model_path}")

# Quick load test
rf_loaded = joblib.load(model_path)
sample_score = rf_loaded.predict_proba(X_test.head(3))[:, 1]
print(f"Load test — sample scores: {sample_score.round(4)}")


## 8. Summary

| Model | CV AUC | Test AUC | Notes |
|---|---|---|---|
| Logistic Regression | — | — | Baseline, interpretable, fast |
| Random Forest | — | — | Best performance, explained by SHAP |

**Business interpretation:**
- The **Critical** tier (score > 0.60) captures the majority of actual defaults with a small fraction of the total portfolio
- `weighted_late_score` and `debt_ratio` are the top two drivers — aligning with Finance intuition
- SHAP waterfall plots can be used per customer to explain individual decisions to stakeholders

**Artifacts produced:**
- `reports/roc_pr_curves.png`
- `reports/confusion_matrix.png`
- `reports/shap_summary.png`
- `reports/shap_bar.png`
- `reports/rf_model.joblib`
- `data/processed/risk_scores.csv`
